# Notebook 01 — Common Table Expressions (CTEs) and Subqueries

**Session:** Database - SQL (3)  
**Module:** 02 · Calculus and Linear Algebra

## Goal
Load `../data/sql_store_seed.sql` into an in-memory SQLite database and master:
1. Finding top-ranked aggregates with `JOIN` vs. **Common Table Expressions (`WITH ... AS`)**.
2. Chaining multiple CTEs into modular analytical pipelines.
3. Scalar, derived-table, and **correlated subqueries** (`SELECT`, `FROM`, `WHERE`).
4. Avoiding the `NOT IN` three-valued logic (`NULL`) trap with `NOT EXISTS`.
5. Expressing universal quantification (**relational division**) with `HAVING COUNT(DISTINCT ...)`.

In [ ]:
import sqlite3
from pathlib import Path

SEED_PATH = Path("../data/sql_store_seed.sql")
conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")
conn.executescript(SEED_PATH.read_text(encoding="utf-8"))


def query(sql: str, params: tuple = (), connection: sqlite3.Connection = conn) -> list[dict]:
    """Execute a SQL statement and return rows as a list of dictionaries."""
    cur = connection.execute(sql, params)
    cols = [desc[0] for desc in cur.description]
    return [dict(zip(cols, row, strict=True)) for row in cur.fetchall()]


def show_rows(rows: list[dict], limit: int | None = None) -> None:
    """Pretty-print query results as an aligned ASCII table."""
    display_rows = rows if limit is None else rows[:limit]
    if not display_rows:
        print("(0 rows)")
        return
    cols = list(display_rows[0].keys())
    widths = {c: max(len(c), *(len(str(r[c])) for r in display_rows)) for c in cols}
    header = " | ".join(f"{c:<{widths[c]}}" for c in cols)
    sep = "-+-".join("-" * widths[c] for c in cols)
    print(header)
    print(sep)
    for r in display_rows:
        print(" | ".join(f"{r[c]!s:<{widths[c]}}" for c in cols))


print("Loaded tables:")
show_rows(query("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;"))

## 1. Highest-Invoiced Client: Direct `JOIN` vs. CTE vs. Subquery

Let us solve the core lecture question three ways:
*Using the invoicing tables, find the client with the highest total invoice amount.*

In [ ]:
# Method A: Direct JOIN + GROUP BY + ORDER BY + LIMIT 1
join_res = query("""
SELECT
    c.client_id,
    c.name,
    ROUND(SUM(i.invoice_total), 2) AS invoice_amount
FROM clients c
JOIN invoices i
    USING (client_id)
GROUP BY c.client_id, c.name
ORDER BY invoice_amount DESC
LIMIT 1;
""")

# Method B: Common Table Expression (CTE)
cte_res = query("""
WITH invoice_amount AS (
    SELECT
        client_id,
        ROUND(SUM(invoice_total), 2) AS invoice_amount
    FROM invoices
    GROUP BY client_id
    ORDER BY invoice_amount DESC
    LIMIT 1
)
SELECT
    c.client_id,
    c.name,
    ia.invoice_amount
FROM clients c
JOIN invoice_amount ia
    USING (client_id);
""")

# Method C: Correlated scalar subquery in SELECT
subquery_res = query("""
SELECT
    c.client_id,
    c.name,
    (
        SELECT ROUND(SUM(i.invoice_total), 2)
        FROM invoices i
        WHERE i.client_id = c.client_id
    ) AS invoice_amount
FROM clients c
ORDER BY invoice_amount DESC
LIMIT 1;
""")

show_rows(cte_res)
assert join_res == cte_res == subquery_res
assert cte_res[0]["client_id"] == 5
assert cte_res[0]["name"] == "Topico Systems"
assert cte_res[0]["invoice_amount"] == 344.68

## 2. Chaining Multiple CTEs

When an analysis requires sequential transformations (e.g., compute per-order totals, compute the global average across orders, then filter orders above average), chained CTEs make each stage explicit and readable.

In [ ]:
above_avg_orders = query("""
WITH order_totals AS (
    SELECT
        order_id,
        ROUND(SUM(quantity * unit_price), 2) AS order_total
    FROM order_items
    GROUP BY order_id
),
global_avg AS (
    SELECT ROUND(AVG(order_total), 2) AS avg_total
    FROM order_totals
)
SELECT
    ot.order_id,
    ot.order_total,
    ga.avg_total
FROM order_totals ot
CROSS JOIN global_avg ga
WHERE ot.order_total > ga.avg_total
ORDER BY ot.order_id;
""")

print(f"Orders above global average: {len(above_avg_orders)}")
show_rows(above_avg_orders)
assert [r["order_id"] for r in above_avg_orders] == [3, 4, 6, 7, 10]

## 3. Correlated Subqueries vs. Rewritten CTEs

Compute each client's average invoice total using a correlated subquery, then compare against a `LEFT JOIN` with a CTE.

In [ ]:
# Correlated scalar subquery
avg_sub = query("""
SELECT
    c.client_id,
    c.name,
    (
        SELECT ROUND(AVG(i.invoice_total), 2)
        FROM invoices i
        WHERE i.client_id = c.client_id
    ) AS avg_invoice
FROM clients c
ORDER BY c.client_id;
""")

# Equivalent CTE + LEFT JOIN
avg_cte = query("""
WITH client_avgs AS (
    SELECT
        client_id,
        ROUND(AVG(invoice_total), 2) AS avg_invoice
    FROM invoices
    GROUP BY client_id
)
SELECT
    c.client_id,
    c.name,
    ca.avg_invoice
FROM clients c
LEFT JOIN client_avgs ca
    ON c.client_id = ca.client_id
ORDER BY c.client_id;
""")

show_rows(avg_sub)
assert avg_sub == avg_cte
assert avg_sub[4]["avg_invoice"] == 172.34  # client 5: (175.32 + 169.36) / 2

## 4. The `NOT IN` Three-Valued Logic (`NULL`) Trap vs. `NOT EXISTS`

If a subquery inside `NOT IN (...)` returns even a single `NULL` value, SQL evaluates `x != NULL` as `UNKNOWN`, causing the entire `NOT IN` filter to return **zero rows**. `NOT EXISTS` never suffers from `NULL` poisoning.

In [ ]:
# Safe anti-join with NOT IN when subquery column is NOT NULL
unordered_not_in = query("""
SELECT product_id, name
FROM products
WHERE product_id NOT IN (
    SELECT product_id
    FROM order_items
)
ORDER BY product_id;
""")

# Demonstrate the NULL trap using orders.shipper_id (which contains NULLs!)
shippers_not_in_trap = query("""
SELECT shipper_id, name
FROM shippers
WHERE shipper_id NOT IN (
    SELECT shipper_id
    FROM orders
);
""")

# Safe alternative: NOT EXISTS
unused_shippers_safe = query("""
SELECT s.shipper_id, s.name
FROM shippers s
WHERE NOT EXISTS (
    SELECT 1
    FROM orders o
    WHERE o.shipper_id = s.shipper_id
)
ORDER BY s.shipper_id;
""")

print("Unordered products:")
show_rows(unordered_not_in)
print(
    f"\nShippers returned by naive NOT IN (with NULLs in orders.shipper_id): {len(shippers_not_in_trap)}"
)
print("Shippers returned by NOT EXISTS:")
show_rows(unused_shippers_safe)

assert [r["product_id"] for r in unordered_not_in] == [7, 12]
assert len(shippers_not_in_trap) == 0  # NULL poisoning!
assert [r["shipper_id"] for r in unused_shippers_safe] == [5]

## 5. Relational Division (`FOR ALL`)

Find customers who have ordered **every** product in the catalog at least once. On the unmodified seed dataset, products 7 and 12 have never been ordered, so 0 customers qualify. Let us verify that, then test inside a temporary transaction where one customer orders every remaining product.

In [ ]:
division_sql = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.customer_id, c.first_name, c.last_name
HAVING COUNT(DISTINCT oi.product_id) = (SELECT COUNT(*) FROM products);
"""

assert len(query(division_sql)) == 0

# Verify relational division logic inside a rolled-back savepoint
conn.execute("SAVEPOINT test_division;")
conn.execute(
    "INSERT INTO orders (order_id, customer_id, order_date, status) VALUES (999, 6, '2026-11-26', 1);"
)
missing_products = query("""
SELECT p.product_id
FROM products p
WHERE NOT EXISTS (
    SELECT 1
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id = 6 AND oi.product_id = p.product_id
);
""")
for row in missing_products:
    conn.execute(
        "INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES (999, ?, 1, 1.00);",
        (row["product_id"],),
    )

qualified = query(division_sql)
show_rows(qualified)
assert len(qualified) == 1
assert qualified[0]["customer_id"] == 6
conn.execute("ROLLBACK TO SAVEPOINT test_division;")
conn.execute("RELEASE SAVEPOINT test_division;")